In [1]:
!pip install scipy

In [2]:
import pandas as pd

from scipy.stats import ttest_ind
from scipy.stats import f_oneway
from scipy.stats import chi2_contingency

In [3]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

In [5]:
df_6 = pd.read_csv("cleaned/olist_orders_dataset_cleaned.csv")

In [7]:
df_6["delivery_delay"].value_counts().head(10)

KeyError: 'delivery_delay'

In [8]:
df_6.columns

Index(['Unnamed: 0', 'order_id', 'customer_id', 'order_status',
       'order_delivered_customer_date', 'order_estimated_delivery_date',
       'order_purchase_Time', 'order_purchase_Date', 'order_approved_at_Time',
       'order_approved_at_Date', 'order_delivered_customer_time'],
      dtype='object')

In [9]:
df_6[[
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]].head()

,order_delivered_customer_date,order_estimated_delivery_date
0,2017-10-10,2017-10-18
1,2018-08-07,2018-08-13
2,2018-08-17,2018-09-04
3,2017-12-02,2017-12-15
4,2018-02-16,2018-02-26


In [10]:
df_6["delivery_days"] = (
    pd.to_datetime(df_6["order_delivered_customer_date"])
    - pd.to_datetime(df_6["order_purchase_Date"])
).dt.days

In [11]:
df_6["delivery_delay"] = (
    pd.to_datetime(df_6["order_delivered_customer_date"])
    - pd.to_datetime(df_6["order_estimated_delivery_date"])
).dt.days

In [12]:
df_6[[
    "delivery_days",
    "delivery_delay"
]].head()

,delivery_days,delivery_delay
0,8.0,-8.0
1,14.0,-6.0
2,9.0,-18.0
3,14.0,-13.0
4,3.0,-10.0


In [13]:
df_6["delivery_delay"].value_counts().head(10)

delivery_delay
-14.0    7126
-13.0    5963
-15.0    5345
-7.0     4837
-8.0     4828
-10.0    4646
-9.0     4626
-11.0    4619
-12.0    4556
-16.0    3903
Name: count, dtype: int64

In [ ]:
df_6.to_csv("orders_feature_engineered.csv", index=False) 

In [15]:
df_6["delivery_status"] = df_6["delivery_delay"].apply(
    lambda x: "Delayed" if x > 0 else "On Time"
)

In [16]:
df_6["delivery_status"].value_counts()

delivery_status
On Time    92906
Delayed     6535
Name: count, dtype: int64

In [ ]:
#T-Test

In [ ]:
#H₀ (Null Hypothesis):  There is no significant difference in the average review score between delayed and on-time orders.
#H₁ (Alternative Hypothesis): There is a significant difference in the average review score between delayed and on-time orders.

In [18]:
df_5 = pd.read_csv("cleaned/olist_order_reviews_dataset_cleaned.csv")

In [19]:
ttest_data = pd.merge(
    df_6[["order_id", "delivery_status"]],
    df_5[["order_id", "review_score"]],
    on="order_id",
    how="inner"
)

In [20]:
ttest_data = ttest_data.dropna(
    subset=["delivery_status", "review_score"]
)

In [21]:
ttest_data["review_score"] = pd.to_numeric(
    ttest_data["review_score"],
    errors="coerce"
)

In [22]:
ttest_data.head()

,order_id,delivery_status,review_score
0,e481f51cbdc54678b7cc49136f2d6af7,On Time,4
1,53cdb2fc8bc7dce0b6741e2150273451,On Time,4
2,47770eb9100c2d0c44946d9cf07ec65d,On Time,5
3,949d5b44dbf5de918fe9c16f97b45f8a,On Time,5
4,ad21c59c0840e6cb83a9ceb5573f8159,On Time,5


In [23]:
ttest_data["delivery_status"].value_counts()

delivery_status
On Time    92814
Delayed     6410
Name: count, dtype: int64

In [24]:
on_time_reviews = ttest_data[
    ttest_data["delivery_status"] == "On Time"
]["review_score"]

delayed_reviews = ttest_data[
    ttest_data["delivery_status"] == "Delayed"
]["review_score"]

In [25]:
print("On-time orders:", len(on_time_reviews))
print("Delayed orders:", len(delayed_reviews))

print(
    "Average review score - On Time:",
    on_time_reviews.mean()
)

print(
    "Average review score - Delayed:",
    delayed_reviews.mean()
)

On-time orders: 92814
Delayed orders: 6410
Average review score - On Time: 4.211789169737324
Average review score - Delayed: 2.271138845553822


In [26]:
t_statistic, t_p_value = ttest_ind(
    delayed_reviews,
    on_time_reviews,
    equal_var=False
)

print("T-statistic:", t_statistic)
print("P-value:", t_p_value)

T-statistic: -96.84922063832963
P-value: 0.0


In [27]:
if t_p_value < 0.05:
    print("Decision: Reject the null hypothesis.")
else:
    print("Decision: Fail to reject the null hypothesis.")

Decision: Reject the null hypothesis.


In [ ]:
#One-Way ANOVA

In [ ]:
#H₀ (Null Hypothesis): The average order value is the same across all product categories.
#H₁ (Alternative Hypothesis):At least one product category has a different average order value.

In [48]:
from scipy.stats import f_oneway

In [32]:
df_3 = pd.read_csv("cleaned/olist_order_items_dataset_cleaned.csv")

In [33]:
df_3.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,13:57:51,199.90,18.14


In [35]:
df_7 = pd.read_csv("cleaned/olist_products_dataset_cleaned.csv")

In [36]:
df_7.head()

,Unnamed: 0,product_id,product_category_name,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,1.0,225.0,16.0,10.0,14.0
1,1,3aa071139cb16b67ca9e5dea641aaa2f,artes,1.0,1000.0,30.0,18.0,20.0
2,2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,1.0,154.0,18.0,9.0,15.0
3,3,cef67bcfe19066a932b7673e239eb23d,bebes,1.0,371.0,26.0,4.0,26.0
4,4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,4.0,625.0,20.0,17.0,13.0


In [37]:
anova_df = pd.merge(
    df_3,
    df_7[["product_id", "product_category_name"]],
    on="product_id",
    how="inner"
)

anova_df.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,09:45:35,58.90,13.29,cool_stuff
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,11:05:13,239.90,19.93,pet_shop
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,14:48:30,199.00,17.87,moveis_decoracao
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,10:10:18,12.99,12.79,perfumaria
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,13:57:51,199.90,18.14,ferramentas_jardim


In [38]:
anova_df["item_spending"] = (
    anova_df["price"] + anova_df["freight_value"]
)

anova_df[[
    "product_category_name",
    "price",
    "freight_value",
    "item_spending"
]].head()

,product_category_name,price,freight_value,item_spending
0,cool_stuff,58.90,13.29,72.19
1,pet_shop,239.90,19.93,259.83
2,moveis_decoracao,199.00,17.87,216.87
3,perfumaria,12.99,12.79,25.78
4,ferramentas_jardim,199.90,18.14,218.04


In [39]:
anova_df["product_category_name"].value_counts()

product_category_name
cama_mesa_banho                  11115
beleza_saude                      9670
esporte_lazer                     8641
moveis_decoracao                  8334
informatica_acessorios            7827
                                 ...  
cds_dvds_musicais                   14
la_cuisine                          14
pc_gamer                             9
fashion_roupa_infanto_juvenil        8
seguros_e_servicos                   2
Name: count, Length: 73, dtype: int64

In [40]:
anova_df["product_category_name"].value_counts().sort_values().head(15)

product_category_name
seguros_e_servicos                                2
fashion_roupa_infanto_juvenil                     8
pc_gamer                                          9
cds_dvds_musicais                                14
la_cuisine                                       14
portateis_cozinha_e_preparadores_de_alimentos    15
artes_e_artesanato                               24
casa_conforto_2                                  30
fashion_esporte                                  30
flores                                           33
musica                                           38
moveis_colchao_e_estofado                        38
fraldas_higiene                                  39
artigos_de_festas                                43
fashion_roupa_feminina                           48
Name: count, dtype: int64

In [41]:
category_counts = anova_df["product_category_name"].value_counts()

anova_df_clean = anova_df[
    anova_df["product_category_name"].isin(
        category_counts[category_counts >= 10].index
    )
]

anova_df_clean.shape

(111028, 9)

In [42]:
anova_df_clean["product_category_name"].nunique()

70

In [43]:
category_groups = [
    group["item_spending"].values
    for name, group in anova_df_clean.groupby("product_category_name")
]

In [44]:
len(category_groups)

70

In [46]:
f_stat, p_value = f_oneway(*category_groups)

print("F-statistic:", f_stat)
print("P-value:", p_value)

F-statistic: 198.81183797128006
P-value: 0.0


In [47]:
alpha = 0.05

if p_value < alpha:
    print("Decision: Reject the null hypothesis.")
else:
    print("Decision: Fail to reject the null hypothesis.")

Decision: Reject the null hypothesis.


In [ ]:
#Chi-Square Test of Independence

In [ ]:
#H₀: Payment method and order status are independent.
#H₁: Payment method and order status are associated.

In [49]:
df_4 = pd.read_csv("cleaned/olist_order_payments_dataset_cleaned.csv")

In [50]:
df_4[["order_id", "payment_type"]].head()

,order_id,payment_type
0,b81ef226f3fe1789b1e8b2acac839d17,credit_card
1,a9810da82917af2d9aefd1278f1dcfa0,credit_card
2,25e8ea4e93396b6fa0d3dd708e76c1bd,credit_card
3,ba78997921bbcdc1373bb41e913ab953,credit_card
4,42fdf880ba16b47b59251dd489d4441a,credit_card


In [51]:
df_6[["order_id", "order_status"]].head()

,order_id,order_status
0,e481f51cbdc54678b7cc49136f2d6af7,delivered
1,53cdb2fc8bc7dce0b6741e2150273451,delivered
2,47770eb9100c2d0c44946d9cf07ec65d,delivered
3,949d5b44dbf5de918fe9c16f97b45f8a,delivered
4,ad21c59c0840e6cb83a9ceb5573f8159,delivered


In [52]:
chi_df = pd.merge(
    df_4[["order_id", "payment_type"]],
    df_6[["order_id", "order_status"]],
    on="order_id",
    how="inner"
)

chi_df.head()

,order_id,payment_type,order_status
0,b81ef226f3fe1789b1e8b2acac839d17,credit_card,delivered
1,a9810da82917af2d9aefd1278f1dcfa0,credit_card,delivered
2,25e8ea4e93396b6fa0d3dd708e76c1bd,credit_card,delivered
3,ba78997921bbcdc1373bb41e913ab953,credit_card,delivered
4,42fdf880ba16b47b59251dd489d4441a,credit_card,delivered


In [53]:
contingency_table = pd.crosstab(
    chi_df["payment_type"],
    chi_df["order_status"]
)

print(contingency_table)

order_status  approved  canceled  created  delivered  invoiced  processing  \
payment_type                                                                 
boleto               0        95        2      19191        67          70   
credit_card          2       444        3      74586       239         224   
debit_card           0         7        0       1486         6           2   
not_defined          0         3        0          0         0           0   
voucher              0       115        0       5493        13          23   

order_status  shipped  unavailable  
payment_type                        
boleto            209          150  
credit_card       851          446  
debit_card         22            6  
not_defined         0            0  
voucher            84           47  


In [54]:
from scipy.stats import chi2_contingency


In [55]:

chi2, p_value, dof, expected = chi2_contingency(contingency_table)

print("Chi-square:", chi2)
print("P-value:", p_value)
print("Degrees of freedom:", dof)

Chi-square: 677.0831369637666
P-value: 1.2047397488650834e-124
Degrees of freedom: 28


In [56]:
print("Minimum expected frequency:", expected.min())

Minimum expected frequency: 5.775561673372735e-05


In [57]:
chi_df_clean = chi_df[
    chi_df["payment_type"] != "not_defined"
]

contingency_table_clean = pd.crosstab(
    chi_df_clean["payment_type"],
    chi_df_clean["order_status"]
)

print(contingency_table_clean)

order_status  approved  canceled  created  delivered  invoiced  processing  \
payment_type                                                                 
boleto               0        95        2      19191        67          70   
credit_card          2       444        3      74586       239         224   
debit_card           0         7        0       1486         6           2   
voucher              0       115        0       5493        13          23   

order_status  shipped  unavailable  
payment_type                        
boleto            209          150  
credit_card       851          446  
debit_card         22            6  
voucher            84           47  


In [59]:
chi2_clean, p_value_clean, dof_clean, expected_clean = chi2_contingency(
    contingency_table_clean
)

print("Chi-square:", chi2_clean)
print("P-value:", p_value_clean)
print("Degrees of freedom:", dof_clean)

Chi-square: 211.5081809889179
P-value: 1.943172279516854e-33
Degrees of freedom: 21


In [60]:
print("Minimum expected frequency:", expected_clean.min())
print("Number of expected frequencies below 5:", (expected_clean < 5).sum())

Minimum expected frequency: 0.029436962736925195
Number of expected frequencies below 5: 10


In [61]:
chi_df_clean["order_status"].value_counts()

order_status
delivered      100756
shipped          1166
canceled          661
unavailable       649
invoiced          325
processing        319
created             5
approved            2
Name: count, dtype: int64

In [62]:
chi_df_clean["order_status_group"] = chi_df_clean["order_status"].replace({
    "delivered": "Completed",
    "canceled": "Unsuccessful",
    "unavailable": "Unsuccessful",
    "approved": "Other",
    "created": "Other",
    "invoiced": "Other",
    "processing": "Other",
    "shipped": "Other"
})

chi_df_clean["order_status_group"].value_counts()

order_status_group
Completed       100756
Other             1817
Unsuccessful      1310
Name: count, dtype: int64

In [63]:
final_contingency_table = pd.crosstab(
    chi_df_clean["payment_type"],
    chi_df_clean["order_status_group"]
)

print(final_contingency_table)

order_status_group  Completed  Other  Unsuccessful
payment_type                                      
boleto                  19191    348           245
credit_card             74586   1319           890
debit_card               1486     30            13
voucher                  5493    120           162


In [64]:
chi2_final, p_value_final, dof_final, expected_final = chi2_contingency(
    final_contingency_table
)

print("Chi-square:", chi2_final)
print("P-value:", p_value_final)
print("Degrees of freedom:", dof_final)

Chi-square: 124.32349537863155
P-value: 2.011329033149798e-24
Degrees of freedom: 6


In [65]:
print("Minimum expected frequency:", expected_final.min())
print("Number of expected frequencies below 5:", (expected_final < 5).sum())

Minimum expected frequency: 19.281210592686
Number of expected frequencies below 5: 0


In [ ]:
#p-value < 0.05, we reject H₀.